## 1. Important Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd 
import numpy as np 
import seaborn as se 
import matplotlib as plt
import random 

## 2. Createting PySpark Session

In [0]:
spark = SparkSession.builder.appName('Data_Analysis_with_PySpark').getOrCreate()

## 3. Generating Data

In [0]:
names = [
    "Alice", "Bob", "Charlie", "David", "Eve", "Fiona", "George", "Hannah",
    "Ivy", "Jack", "Kaitlyn", "Liam", "Olivia", "Liam", "Emma", "Noah", 
    "Ava", "Oliver", "Charlotte", "Elijah", "Sophia", "James", "Amelia", 
    "Benjamin", "Isabella", "Lucas", "Mia", "Mason", "Harper", "Ethan", 
    "Evelyn", "Alexander", "Abigail", "Henry", "Ella", "Jackson", "Scarlett", 
    "Aiden", "Grace", "Samuel", "Lily", "Sebastian"
]
genders = ["Male", "Female", None]
subjects = ["Math", "Science", "History", "English", "Art", "PE", None]
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", 
    "Bangalore", "Hajipur", "Sitamardhi", "MP", None
]
states = ["NY", "CA", "IL", "TX", "Bihar", "Karnataka", "Sitamardhi", None]
countries = ["USA", "India", "Pakistan", "Nepal", "China", None]
graduated_status = ["Yes", "No", None]

data = [
    (
        i, 
        random.choice(names),  # student_name
        random.choice([random.randint(18, 25), None]),  # age
        random.choice(genders),  # gender
        random.choice(subjects),  # subject
        random.choice([random.randint(50, 100), None]),  # marks
        random.choice(cities),  # city
        random.choice(states),  # state
        random.choice(countries),  # country
        random.choice(graduated_status),  # graduated
    )
    for i in range(1, 501)
]

## 4. Creating a DATAFRAME

In [0]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("marks", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("graduated", StringType(), True),
])

df = spark.createDataFrame(data,schema=schema)
df.show(5)

+----------+------------+----+------+-------+-----+-----------+---------+-------+---------+
|student_id|student_name| age|gender|subject|marks|       city|    state|country|graduated|
+----------+------------+----+------+-------+-----+-----------+---------+-------+---------+
|         1|      Amelia|  18|  null|    Art| null|   New York|       IL|  China|     null|
|         2|     Charlie|  22|  null|Science|   54|    Chicago|Karnataka|  India|      Yes|
|         3|        Noah|null|Female|History|   56|Los Angeles|       CA|    USA|     null|
|         4|    Isabella|null|  null|History| null|         MP|Karnataka|  China|       No|
|         5|       Fiona|null|  null|    Art|   85|  Bangalore|Karnataka|  India|      Yes|
+----------+------------+----+------+-------+-----+-----------+---------+-------+---------+
only showing top 5 rows



## 5.OverView of DataFrame

In [0]:
df.show(10)

+----------+------------+----+------+-------+-----+-----------+---------+--------+---------+
|student_id|student_name| age|gender|subject|marks|       city|    state| country|graduated|
+----------+------------+----+------+-------+-----+-----------+---------+--------+---------+
|         1|      Amelia|  18|  null|    Art| null|   New York|       IL|   China|     null|
|         2|     Charlie|  22|  null|Science|   54|    Chicago|Karnataka|   India|      Yes|
|         3|        Noah|null|Female|History|   56|Los Angeles|       CA|     USA|     null|
|         4|    Isabella|null|  null|History| null|         MP|Karnataka|   China|       No|
|         5|       Fiona|null|  null|    Art|   85|  Bangalore|Karnataka|   India|      Yes|
|         6|       Mason|null|  null|   Math| null|       null|       NY|   China|      Yes|
|         7|      Hannah|  21|Female|   null|  100|         MP|    Bihar|Pakistan|      Yes|
|         8|      Elijah|  25|  null|     PE| null|    Chicago|       

In [0]:
df.describe().show()

+-------+-----------------+------------+------------------+------+-------+-----------------+----------+-----+-------+---------+
|summary|       student_id|student_name|               age|gender|subject|            marks|      city|state|country|graduated|
+-------+-----------------+------------+------------------+------+-------+-----------------+----------+-----+-------+---------+
|  count|              500|         500|               259|   313|    415|              253|       458|  446|    419|      320|
|   mean|            250.5|        null|21.571428571428573|  null|   null|74.79841897233202|      null| null|   null|     null|
| stddev|144.4818327679989|        null| 2.239037524595077|  null|   null|14.20746497382611|      null| null|   null|     null|
|    min|                1|     Abigail|                18|Female|    Art|               50| Bangalore|Bihar|  China|       No|
|    max|              500|      Sophia|                25|  Male|Science|              100|Sitamardhi| 

In [0]:
df.dtypes

Out[43]: [('student_id', 'int'),
 ('student_name', 'string'),
 ('age', 'int'),
 ('gender', 'string'),
 ('subject', 'string'),
 ('marks', 'int'),
 ('city', 'string'),
 ('state', 'string'),
 ('country', 'string'),
 ('graduated', 'string')]

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- graduated: string (nullable = true)



In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,187,85,0,42,54,81,180


## 6. Replacing Null Values

In [0]:
df = df.fillna({
    'gender' : 'UniSex',
    'subject' : 'Hindi',
    'city' : 'Kalitand',
    'State' : 'Others',
    'Country' : 'India',
    'graduated' : 'Failed'
})

In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,0,0,0,0,0,0,0


## 7. Checking duplicate Values in each column

In [0]:
for index,column in enumerate(df.columns):
    print(f"checking duplicates in the column: {column}")
    duplicate_count = df.groupBy(column).count().filter("count > 1 ")
    duplicate_count.show()


checking duplicates in the column: student_id
+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+

checking duplicates in the column: student_name
+------------+-----+
|student_name|count|
+------------+-----+
|       Grace|   11|
|       Lucas|   10|
|         Ivy|   13|
|    Isabella|   12|
|    Benjamin|    9|
|      Hannah|    9|
|        Jack|    9|
|        Ella|   10|
|      Evelyn|   15|
|        Noah|   15|
|       Mason|   14|
|       Ethan|   14|
|     Charlie|   16|
|         Mia|   12|
|         Bob|   11|
|        Liam|   23|
|   Sebastian|   14|
|      Elijah|    9|
|      Samuel|   11|
|   Alexander|    9|
+------------+-----+
only showing top 20 rows

checking duplicates in the column: age
+----+-----+
| age|count|
+----+-----+
|  22|   43|
|null|  241|
|  20|   29|
|  23|   34|
|  25|   33|
|  24|   28|
|  21|   32|
|  18|   30|
|  19|   30|
+----+-----+

checking duplicates in the column: gender
+------+-----+
|gender|count|
+------+-----+
|Fe

In [0]:
duplicate_counts = []
# Loop through each column in the DataFrame
for column in df.columns:
    # Group by the column and count duplicates (count > 1)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,50
6,city,9
7,state,8
8,country,5
9,graduated,3
